<a href="https://colab.research.google.com/github/allaalmouiz/MedBot_LoRa/blob/main/MedBot_on_Custom_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#MedBot built on custom dataset
Submitted by: **`Alaa Almouiz F. Moh.`**

ID Number: **`S2026_176`**

Track: **Machine Learning**

For: **ZAKA AI, Inc. All Rights Reserved.©**

## **Problem Statement (Objective)**
The objective of this project is to create a simple QA LLM that can answer medical questionsand customize this LLM with any dataset.

**Just to give you a heads up:** We won't be having a model performing like ChatGPT or Bard, but at least we will have an idea about how we can create our own smaller versions of such powerful LLMs.  

## Importing and Installing Libraries/Packages
We will start by installing our necessary packages.

**bitsandbytes**: This package will allow us to run 4bit quantization on our model

**transformers**: This Hugging Face package will allow us to load state-of-the-art models easily into our notebook

**peft**: This package allows us to add PEFT techniques easily to our model, such as LoRA

**accelerate**: Accelerate is a handy package that allows us to run boiler plate code with a few lines of code

**datasets**: This package allows us to easily import datasets from the Hugging Face platform to be directly used

In [ ]:
!pip install bitsandbytes
!pip install git+https://github.com/huggingface/transformers.git
!pip install git+https://github.com/huggingface/peft.git
!pip install git+https://github.com/huggingface/accelerate.git
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00
  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-i139_97l
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-i139_97l
  Resolved https://github.com/huggingface/transformers.git to commit d64a6d67d8c004a25570db4df5689e06caea6af7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.3.0.dev0-py3-none-any.whl size=11327720 sha256=4fa7af557b64d3203e409d225db055f75844ef6f41eddb66c447702777e3756e
  Stored in directory: /tmp/pip-ephem-wheel-cache-oq6pnyim/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
  

In [ ]:
import torch
import transformers
from peft import prepare_model_for_kbit_training
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM

## Loading our model

Let's start by loading our model. We will use the GPT Neox 20b Model by EleutherAI!

I changed the model to `EleutherAI/gpt-neo-1.3b` model since my GPU RAM won't

In [ ]:
hf_model = "EleutherAI/gpt-neo-1.3b"

We will also set the bitsandbytes configurations needed for our model to run on our single colab GPU. The needed paramaters will be 'Double Quantization' 'Quantization Type' and the computational type needs to be set to bfloat16.

In [ ]:
bitsbytes_config = BitsAndBytesConfig(load_in_4bit=True,
                                      bnb_4bit_use_double_quant=True,
                                      bnb_4bit_quant_type="nf4",
                                      bnb_4bit_compute_dtype=torch.bfloat16)

🔮 **My Notes**

**Configuration of the `BitsAndBytesConfig`**:  *Things I used*
* `load_in_4bit=True` - Enabling 4 bits Quantization (Trade-off between Size/Speed)
* `bnb_4bit_use_double_quant=True` - Applying second layer of quant to already quantized weights: "nested quantization"
* `bnb_4bit_compute_dtype=torch.float16` - Half precession for computation, high speed.
* `bnb_4bit_quant_type="nf4"` - Normal Float4

We will then set our tokenizer, and our model using the AutoTokenizer and AutoModelforCausalLM classes

In [ ]:
# Tokenizer intialization
tokenizer = AutoTokenizer.from_pretrained(hf_model)
tokenizer.pad_token = tokenizer.eos_token # Set the padding token here

# Model intialization
model = AutoModelForCausalLM.from_pretrained(hf_model,
                                             device_map = 'auto',
                                             quantization_config= bitsbytes_config)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.31G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-1.3b
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0...23}.attn.attention.masked_bias | UNEXPECTED |  | 
transformer.h.{0...22}.attn.attention.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


I faced a `OutOfMemoryError` and i cant update my colab. so I will be using another lighter model to complete the experiment.



The model -> `EleutherAI/gpt-neo-1.3b`

🔮 **My Notes**

* For the `tokenizer` I used the `AutoTokenizer` from pre-trained for my hugging face model, and applied End of sequence padding as well to it.
* For the model, I used the `AutoModelCausalLM` from pre-trained with my `bitsbytes_config` I early configured -> 4 bits.

## Model Preprocessing

We now have to apply some preprocessing to our model so we can prepare it for training. First we need to further reduce our memory consumption by using the gradient_checkpointing_enable() fucntion on our model. We then use the prepare_model_for_kbit_training function so that we can use 4bit quantization training.

In [ ]:
#Test Your Zaka

model.gradient_checkpointing_enable()           # Reduces memory by recomputing activations
model = prepare_model_for_kbit_training(model)  # Prepares model for 4-bit training

Explain with your own words how 4-bit quantization affects accuracy.

**Test your Zaka**

🔮 **My Notes**

**Quantization** offers a solution to efficiently load and use large LLMs without compromising performance -> decrease high-precision weights and activations -> decrease Memory used.

Actually since the precicion of trainable model parameters is decreased then this would affect accuracy a bit but not so much since we will trade with **decreasing memory usage** and **increase speed**.

This is done through 4bit quantization as in here and also 8bit quantization and also we could double the quantization.


We will also set a function that will print the number of trainable parameters our model has.

In [ ]:
def print_trainable_parameters(model):
    trainable_parameters = 0
    all_paramaters = 0
    for _, param in model.named_parameters():
        all_paramaters += param.numel()
        if param.requires_grad:
            trainable_parameters += param.numel()
    print(
        f"Trainable: {trainable_parameters} || All: {all_paramaters} || Trainable %: {100 * trainable_parameters / all_paramaters}"
    )

In [ ]:
# Print all named modules to find the right ones
for name, module in model.named_modules():
    print(name)



transformer
transformer.wte
transformer.wpe
transformer.drop
transformer.h
transformer.h.0
transformer.h.0.ln_1
transformer.h.0.attn
transformer.h.0.attn.attention
transformer.h.0.attn.attention.attn_dropout
transformer.h.0.attn.attention.resid_dropout
transformer.h.0.attn.attention.k_proj
transformer.h.0.attn.attention.v_proj
transformer.h.0.attn.attention.q_proj
transformer.h.0.attn.attention.out_proj
transformer.h.0.ln_2
transformer.h.0.mlp
transformer.h.0.mlp.c_fc
transformer.h.0.mlp.c_proj
transformer.h.0.mlp.act
transformer.h.0.mlp.dropout
transformer.h.1
transformer.h.1.ln_1
transformer.h.1.attn
transformer.h.1.attn.attention
transformer.h.1.attn.attention.attn_dropout
transformer.h.1.attn.attention.resid_dropout
transformer.h.1.attn.attention.k_proj
transformer.h.1.attn.attention.v_proj
transformer.h.1.attn.attention.q_proj
transformer.h.1.attn.attention.out_proj
transformer.h.1.ln_2
transformer.h.1.mlp
transformer.h.1.mlp.c_fc
transformer.h.1.mlp.c_proj
transformer.h.1.mlp.ac

Finally we will set the configurations for our LoRA. The paramaters needed are the rank updates, the default LoRa alpha value, the target modules which need to be set to query_key_value, the default lora dropout rate, bias should be set to none, and the task type according to the model we are using.

In [ ]:
config = LoraConfig(
    #Test Your Zaka
    r = 16,
    lora_alpha = 32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj'], # Corrected target modules for GPT-Neo
    lora_dropout=0.05, #reg
    bias= "none",
    task_type = "CAUSAL_LM"
)

# Insert the configs above to the model using the get_peft_model function
#Test Your Zaka
model = get_peft_model(model, config)

# Print the trainable parameters of the model
print_trainable_parameters(model)

Trainable: 6291456 || All: 717887488 || Trainable %: 0.8763846849494054


## Dataset Loading

Let's load our medical dataset from Hugging Face. We will use the `medalpaca/medical_meadow_wikidoc_patient_information` dataset. You can access it [here](https://huggingface.co/datasets/medalpaca/medical_meadow_wikidoc).

In [ ]:
#Test Your Zaka

# Loading the datset
data = load_dataset("medalpaca/medical_meadow_wikidoc_patient_information")

# Mapping the needed column as our data using a lambda statement
data = data.map(lambda samples: tokenizer(samples["output"],
                max_length=256,
                padding="max_length",
                truncation  =True),
                batched=True)

README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc_patient_info.json:   0%|          | 0.00/3.49M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5942 [00:00<?, ? examples/s]

Map:   0%|          | 0/5942 [00:00<?, ? examples/s]

## Model Training and Testing

Now we train the model usig the transformers library. Before doing so, we set the tokenizer to be the end of sequence tokens since it is required by our model. Your goal here is to tune the paramaters until you get a running model on a single colab GPU.

In [ ]:
# Setting the tokenizer padding to be 'eos' tokens
tokenizer.pad_token = tokenizer.eos_token

training_args = transformers.TrainingArguments(
    gradient_accumulation_steps=8, # Accumulate gradients over 8 steps (simulates larger batch)
    output_dir = "./results",
    per_device_train_batch_size = 1,
    num_train_epochs=3,
    logging_steps = 1,
    save_total_limit = 1, # keep the latest checkpoint
    learning_rate=0.001,
    max_steps = 40,
    warmup_steps=2,
    fp16 = True,
    max_grad_norm=1.0 # Add gradient clipping to prevent exploding gradients
)

trainer = transformers.Trainer(
    model = model,
    args = training_args,
    train_dataset = data["train"],
    processing_class = tokenizer,
    data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

# This silences the warnings
model.config.use_cache = False

# Train the model!
#Test Your Zaka
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000
5,0.000000
6,0.000000
7,0.000000
8,0.000000
9,0.000000
10,0.000000


TrainOutput(global_step=40, training_loss=0.0, metrics={'train_runtime': 164.3882, 'train_samples_per_second': 1.947, 'train_steps_per_second': 0.243, 'total_flos': 597072260628480.0, 'train_loss': 0.0, 'epoch': 0.05385392123864019})

Explain 4 of the training arguments you used in your Trainer, how they are used, and what do they represent

**Test your Zaka**

🔮 **My Notes:**

I used arguments for my trainig in my `Trainer` inside the `training_args`:
* `save_total_limit=1` - to only save the last checkpoint to save since there're memory constarints.

* `gradient_accumulation_steps=8` - this is to accumulate the gradient update and change only after 8 steps -> similar to having larger batches.

* `fp16 = True` - Enabling 16bit trainig for efficency and  memory usage.

* `warmup_steps=2`- to gradually warm up the LR before jumping into the full LR which is `learning_rate=2e-4`

We now save our model as a pretrained version so that we can set the LoRA configurations. This model will be saved to a separate folder on the next block.

In [ ]:
#Test Your Zaka

trainer.save_model("outputs")
saved_model = model.merge_and_unload() if hasattr(model, "merge_and_unload") else model
saved_model.save_pretrained("outputs")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Before testing our model, we have to get the LoRA configs from our ..dvpre-trained model and set them to our new model using the get_peft_model() function.

In [ ]:
#Test Your Zaka

lora_configs = LoraConfig.from_pretrained("outputs")
model = get_peft_model(model, lora_configs)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:78: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'EleutherAI/gpt-neo-1.3b' to 'None'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:301: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


We need to set our prompt as a variable, and also our device currently in use.

In [ ]:
#Test Your Zaka

prompt = "What are the symptoms of Flu?"
device = "cuda:0"

Finally, we will make our LLM generate text based on the data. First we user the tokenizer() function on our prompt.

In [ ]:
#Test Your Zaka
inputs = tokenizer(prompt, return_tensors="pt").to(device)

Let's now use the generate() function on our model, and print the decoded version of our output.

In [ ]:
outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

What are the symptoms of Flu? Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown Showdown
